In [8]:
import pandas as pd

In [ ]:
# --- 1. DATA PREPARATION ---
df = pd.read_csv('../data/v4/All_Data_Final.csv')
if 'Unnamed: 0' in df.columns:
    df.drop('Unnamed: 0', axis=1, inplace=True)
if 'Unnamed: 0.1' in df.columns:
    df.drop('Unnamed: 0.1', axis=1, inplace=True)

df

,Game_ID,Season,Matchweek,Game_Type,Time,Team,Home_Score,Away_Score,Home_Red_Count,Away_Red_Count,...,Away_Sub_Count,Event_Type,Player_1,Pos_1,Player_2,Pos_2,Note,Odds_Home_Win,Odds_Draw,Odds_Away_Win
0,120230101,2023,1,League,20,Away,0,1,0,0,...,0,Goal,Isi Palazón,NaN,NaN,NaN,Goal,2.63,3.16,2.87
1,120230101,2023,1,League,28,Away,0,2,0,0,...,0,Goal,Randy Ntekja,NaN,NaN,NaN,Goal,2.63,3.16,2.87
2,120230101,2023,1,League,40,Away,0,2,0,0,...,0,YELLOW,Unai López,NaN,NaN,NaN,YELLOW,2.63,3.16,2.87
3,120230101,2023,1,League,46,Away,0,2,0,0,...,1,Substitution,Pathé Ciss,Central Midfield,Unai López,NaN,Neutral Substitution,2.63,3.16,2.87
4,120230101,2023,1,League,46,Home,0,2,0,0,...,1,Substitution,Sergio Arribas,Midfield,Lázaro,NaN,Neutral Substitution,2.63,3.16,2.87
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85491,520253403,2025,34,League,77,Away,0,2,0,0,...,4,Substitution,Paulo Dybala,Offence,Donyell Malen,NaN,Offensive Substitution,2.95,3.21,2.43
85492,520253403,2025,34,League,77,Home,0,2,0,0,...,4,Substitution,Nicolò Cambiaghi,Offence,Riccardo Orsolini,NaN,Offensive Substitution,2.95,3.21,2.43
85493,520253403,2025,34,League,77,Home,0,2,0,0,...,4,Substitution,Martin Vitík,Defence,Eivind Helland,NaN,Neutral Substitution,2.95,3.21,2.43
85494,520253403,2025,34,League,90,Away,0,2,0,0,...,5,Substitution,Jan Ziółkowski,Defence,Niccolo Pisilli,NaN,Neutral Substitution,2.95,3.21,2.43


In [10]:
# 1. Normalize columns
df['Event_Type'] = df['Event_Type'].fillna('').astype(str).str.upper()
df['Team'] = df['Team'].fillna('').astype(str).str.strip().str.capitalize()
df['Note'] = df['Note'].fillna('').astype(str).str.upper()

# 2. Rename existing sub columns
df = df.rename(columns={
    'Home_Sub_Count': 'Home_Total_Sub_Count',
    'Away_Sub_Count': 'Away_Total_Sub_Count'
})

# 3. Identify Card & Sub events
is_home = df['Team'] == 'Home'
is_away = df['Team'] == 'Away'

# Cards
is_yellow = df['Event_Type'].str.contains('YELLOW')
is_red = (df['Event_Type'] == 'RED') | (df['Event_Type'] == 'YELLOW_RED')

# Subs
is_sub = df['Event_Type'].str.contains('SUBSTITUTION')
is_def = is_sub & df['Note'].str.contains('DEFENSIVE')
is_off = is_sub & df['Note'].str.contains('OFFENSIVE')
is_neu = is_sub & df['Note'].str.contains('NEUTRAL')

# 4. Calculate Rolling Totals (Grouped by Game_ID)
# Cards
df['Home_Yellow_Count'] = (is_yellow & is_home).groupby(df['Game_ID']).cumsum()
df['Away_Yellow_Count'] = (is_yellow & is_away).groupby(df['Game_ID']).cumsum()
df['Home_Red_Count'] = (is_red & is_home).groupby(df['Game_ID']).cumsum()
df['Away_Red_Count'] = (is_red & is_away).groupby(df['Game_ID']).cumsum()

# Substitutions - Home
df['Home_Total_Sub_Count'] = (is_sub & is_home).groupby(df['Game_ID']).cumsum()
df['Home_Defensive_Sub_Count'] = (is_def & is_home).groupby(df['Game_ID']).cumsum()
df['Home_Offensive_Sub_Count'] = (is_off & is_home).groupby(df['Game_ID']).cumsum()
df['Home_Neutral_Sub_Count'] = (is_neu & is_home).groupby(df['Game_ID']).cumsum()

# Substitutions - Away
df['Away_Total_Sub_Count'] = (is_sub & is_away).groupby(df['Game_ID']).cumsum()
df['Away_Defensive_Sub_Count'] = (is_def & is_away).groupby(df['Game_ID']).cumsum()
df['Away_Offensive_Sub_Count'] = (is_off & is_away).groupby(df['Game_ID']).cumsum()
df['Away_Neutral_Sub_Count'] = (is_neu & is_away).groupby(df['Game_ID']).cumsum()

In [11]:
df.head(30)

,Game_ID,Season,Matchweek,Game_Type,Time,Team,Home_Score,Away_Score,Home_Red_Count,Away_Red_Count,...,Note,Odds_Home_Win,Odds_Draw,Odds_Away_Win,Home_Defensive_Sub_Count,Home_Offensive_Sub_Count,Home_Neutral_Sub_Count,Away_Defensive_Sub_Count,Away_Offensive_Sub_Count,Away_Neutral_Sub_Count
0,120230101,2023,1,League,20,Away,0,1,0,0,...,GOAL,2.63,3.16,2.87,0,0,0,0,0,0
1,120230101,2023,1,League,28,Away,0,2,0,0,...,GOAL,2.63,3.16,2.87,0,0,0,0,0,0
2,120230101,2023,1,League,40,Away,0,2,0,0,...,YELLOW,2.63,3.16,2.87,0,0,0,0,0,0
3,120230101,2023,1,League,46,Away,0,2,0,0,...,NEUTRAL SUBSTITUTION,2.63,3.16,2.87,0,0,0,0,0,1
4,120230101,2023,1,League,46,Home,0,2,0,0,...,NEUTRAL SUBSTITUTION,2.63,3.16,2.87,0,0,1,0,0,1
5,120230101,2023,1,League,51,Away,0,2,0,0,...,YELLOW,2.63,3.16,2.87,0,0,1,0,0,1
6,120230101,2023,1,League,63,Away,0,2,0,0,...,NEUTRAL SUBSTITUTION,2.63,3.16,2.87,0,0,1,0,0,2
7,120230101,2023,1,League,65,Home,0,2,0,0,...,YELLOW,2.63,3.16,2.87,0,0,1,0,0,2
8,120230101,2023,1,League,65,Away,0,2,0,0,...,YELLOW,2.63,3.16,2.87,0,0,1,0,0,2
9,120230101,2023,1,League,66,Home,0,2,0,0,...,NEUTRAL SUBSTITUTION,2.63,3.16,2.87,0,0,2,0,0,2


In [13]:
df

,Game_ID,Season,Matchweek,Game_Type,Time,Team,Home_Score,Away_Score,Home_Red_Count,Away_Red_Count,...,Note,Odds_Home_Win,Odds_Draw,Odds_Away_Win,Home_Defensive_Sub_Count,Home_Offensive_Sub_Count,Home_Neutral_Sub_Count,Away_Defensive_Sub_Count,Away_Offensive_Sub_Count,Away_Neutral_Sub_Count
0,120230101,2023,1,League,20,Away,0,1,0,0,...,GOAL,2.63,3.16,2.87,0,0,0,0,0,0
1,120230101,2023,1,League,28,Away,0,2,0,0,...,GOAL,2.63,3.16,2.87,0,0,0,0,0,0
2,120230101,2023,1,League,40,Away,0,2,0,0,...,YELLOW,2.63,3.16,2.87,0,0,0,0,0,0
3,120230101,2023,1,League,46,Away,0,2,0,0,...,NEUTRAL SUBSTITUTION,2.63,3.16,2.87,0,0,0,0,0,1
4,120230101,2023,1,League,46,Home,0,2,0,0,...,NEUTRAL SUBSTITUTION,2.63,3.16,2.87,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85491,520253403,2025,34,League,77,Away,0,2,0,0,...,OFFENSIVE SUBSTITUTION,2.95,3.21,2.43,1,1,1,2,2,0
85492,520253403,2025,34,League,77,Home,0,2,0,0,...,OFFENSIVE SUBSTITUTION,2.95,3.21,2.43,1,2,1,2,2,0
85493,520253403,2025,34,League,77,Home,0,2,0,0,...,NEUTRAL SUBSTITUTION,2.95,3.21,2.43,1,2,2,2,2,0
85494,520253403,2025,34,League,90,Away,0,2,0,0,...,NEUTRAL SUBSTITUTION,2.95,3.21,2.43,1,2,2,2,2,1


In [14]:
# Save the fixed data
df.to_csv('All_Data_Final_Fixed.csv', index=False)